In [ ]:
import numpy as np
import json
import sys, os, importlib, math
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from skimage.segmentation import mark_boundaries
import io, imageio

from matplotlib import rc
rc('text',usetex=True)
rc('text.latex', preamble='\\usepackage{color}')

import shap_bpt as shap_bpt
print('shap_bpt version:',shap_bpt.__version__)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "mathtext.fontset": "cm",
})

In [ ]:
import os
from pathlib import Path
import yaml

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'shap_bpt').is_dir():
            return path
    raise FileNotFoundError('Could not find the project root from the current working directory.')

original_working_dir = Path.cwd()
project_root = find_project_root(original_working_dir)
os.chdir(project_root)
print(f'Project root: {project_root}')

config_file = "MSCOCO_mac"
# config_file = "MSCOCO_xn2"

try:
    with open(project_root / f"examples/configs/{config_file}.yaml", "r") as f:
        config = yaml.safe_load(f)
finally:
    os.chdir(original_working_dir)
    print(f'Restored working directory: {original_working_dir}')

dataset_root = config["data"]["dataset_root"]
print(f"Dataset root: {dataset_root}")

In [ ]:
## Get available precomputed SAM partitions
# fetch available unique image_ids from the partitions directory

# partition_dir = 'partitions_24'
partition_dir = 'partitions_new_120'
image_ids = os.listdir(f"../{partition_dir}")

image_ids = [f.split('.')[0].split('_')[0] for f in image_ids if '_refined.npy' in f]
print(f'computed image_ids: {len(image_ids)}')

# already_computed = ['000000049091',
#  '000000000632',
#  '000000186929',
#  '000000002299',
#  '000000225757',
#  '000000171382']

image_ids = [img_id for img_id in image_ids]
print(f'filtered image_ids: {len(image_ids)}')
print(image_ids[:10])
image_dir = config["data"]["image_dir"] # Update for your image directory

In [ ]:
image_name = "000000171382"

# Get Image index from image_ids list
image_index = image_ids.index(image_name)
image_index


In [ ]:
# image_id = '000000170670'  # Example image ID
image_id =  image_ids[image_index]  # Example image ID
image_path = os.path.join(image_dir, f'{image_id}.jpg')

In [ ]:
image_paths = [os.path.join(image_dir, f'{img_id}.jpg') for img_id in image_ids[:5]]
image_paths

In [ ]:
# import imageio as iio
# from skimage.transform import resize
# import hashlib

def load_rgb_image_from_file(filename):
    return cv2.imread(filename, cv2.IMREAD_COLOR)[:,:,::-1]


In [ ]:
# image_fnames = [
#     '../imgs/flamingo.png',
#     '../imgs/egret.png',
#     '../imgs/bird4.png',
#     '../imgs/sorrel.png',
#     '../imgs/00000001_000.png',
#     '../imgs/lungaca20.jpeg',
#     '../imgs/toucan.png',
# ]

# images = [ cv2.imread(f, cv2.IMREAD_COLOR)[:,:,::-1].astype(np.uint8) for f in image_fnames ]
images = [ load_rgb_image_from_file(f) for f in image_paths ]

In [ ]:
image_path

In [ ]:
image_to_explain = load_rgb_image_from_file(image_path)
print(f"Image shape: {image_to_explain.shape}, dtype: {image_to_explain.dtype}")
plt.imshow(image_to_explain); plt.xticks([]); plt.yticks([]); plt.title(f"Image ID: {image_id}", fontsize=12)
plt.show()

In [ ]:
image_id  = image_path.split('/')[-1].split('.')[0]
image_id

In [ ]:
image_no = int(image_id)
image_no
from pycocotools.coco import COCO

image_dir = config["data"]["image_dir"] # Update for your image directory
annotation_file =  config["data"]["annotation_file"] # Update for your annotation file
coco = COCO(annotation_file)

categories = coco.loadCats(coco.getCatIds())
coco_categories = {cat['id']: cat['name'] for cat in categories}


## Groundtruth

In [ ]:
image_info = coco.loadImgs(image_no)[0]
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)
has_segmentation = any('segmentation' in ann for ann in annotations)
print(f"Segmentation annotations present: {has_segmentation}")

In [ ]:
# categories = coco.loadCats(coco.getCatIds())
# coco_categories = {cat['id']: cat['name'] for cat in categories}
# Get annotations for the selected image
ann_ids = coco.getAnnIds(imgIds=image_info['id'])
annotations = coco.loadAnns(ann_ids)

In [ ]:
# Check if segmentation annotations are present
has_segmentation = any('segmentation' in ann for ann in annotations)
print(f"Segmentation annotations present: {has_segmentation}")
if has_segmentation:
    for ann in annotations:
        if 'segmentation' in ann:
            print(f"Segmentation Annotation: {ann['segmentation']}")
            break

In [ ]:
def get_annotation(coco,image_no,category_name=None):
    if isinstance(image_no, str):
        image_no = int(image_no.split('\\')[-1].split('.')[0])
    
    image_info = coco.loadImgs(image_no)[0]
    if category_name is None:
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'])
    else:
        category_ids = coco.getCatIds(catNms=[category_name])
        annotation_ids = coco.getAnnIds(imgIds=image_info['id'], catIds=category_ids)
    annotations = coco.loadAnns(annotation_ids)
    return annotations

def create_gt(coco,image_no,category_name=None, verbose=False):
    annotations = get_annotation(coco,image_no,category_name=category_name)
    if verbose:
        if len(annotations)>0:
            print(f"Image:{image_info['id']} has {len(annotations)} annotations")
    
    mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)
    category_mask = np.zeros((image_info['height'], image_info['width']), dtype=np.uint8)

    # Combine all masks for this image
    for ann in annotations:
        if 'segmentation' in ann:
            category_id = ann['category_id']  # Unique ID for object category
            print('category_id',category_id)
            # Decode the segmentation mask
            if isinstance(ann['segmentation'], list):  # Polygon format
                for seg in ann['segmentation']:
                    pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(mask, [pts], color=1)  # Fill the mask polygon
                    cv2.fillPoly(category_mask, [pts], color=category_id)
            elif isinstance(ann['segmentation'], dict):  # RLE format
                rle = ann['segmentation']
                decoded_mask = coco.annToMask(ann)
                mask += decoded_mask  # Add binary mask
                category_mask[decoded_mask > 0] = category_id  # Assign category ID
    
    # Resize masks to match actual image dimensions
    if mask.shape[:2] != image_to_explain.shape[:2]:
        # print(f"Resizing masks: Annotated={mask.shape}, Actual={image_to_explain.shape[:2]}")
        mask = cv2.resize(mask, (image_to_explain.shape[1], image_to_explain.shape[0]), interpolation=cv2.INTER_NEAREST)
        category_mask = cv2.resize(category_mask, (image_to_explain.shape[1], image_to_explain.shape[0]), interpolation=cv2.INTER_NEAREST)
    return mask,category_mask,annotations

In [ ]:
def load_groundtruth(coco,image_no,fixed_category=None):
    global ground_truth,weighted_ground_truth,annotations
    mask,ground_truth,annotations = create_gt(coco,image_no)
    # weighted_ground_truth = gaussian_filter(ground_truth.astype(float), 16) * ground_truth
    ground_truth.dtype = 'bool'
# fixed_category = fixed_category
load_groundtruth(coco,image_no)

In [ ]:
fixed_category = None

In [ ]:
plt.imshow(image_to_explain); plt.xticks([]); plt.yticks([]); plt.title(f'Image ID: {image_id}'); plt.show()
plt.show()
plt.imshow(ground_truth, cmap='tab20'); plt.xticks([]); plt.yticks([]); plt.title(f'Ground Truth Mask for {fixed_category}'); plt.show()

In [ ]:
from skimage.segmentation import quickshift

In [ ]:
%%time
print(images[0].shape)
shap_bpt.build_bpt_from_image(images[0])

In [ ]:
%%time
quickshift(images[0])

In [ ]:
%%time
bptrees = [ shap_bpt.build_bpt_from_image(image=img, use_8ways=True#, use_sqrt_area=False#squared_perimeter=False, squared_color=False#cv2.cvtColor(image_to_explain, cv2.COLOR_BGR2LAB), 
#                                           squared_perimeter=False, use_8ways=True, use_lab=True, squared_color=True
           ) for img in images ]

In [ ]:
# %%time
import matplotlib.colors as mcolors
cmap = [shap_bpt.hex_to_rgb(c[1]) for c in list(mcolors.XKCD_COLORS.items())]

def colorize(nodes, img, i):
    is_aa = isinstance(nodes[0], shap_bpt.AxisAlignedSegment)
    pxflat_image = img.reshape((img.shape[0] * img.shape[1], 3))
    colored = np.zeros_like(img, dtype=np.float32)
    flat_colored = colored.reshape(pxflat_image.shape)
    for node in nodes:
        if is_aa:
            clr = np.mean(np.mean(img[node.ymin:node.ymax, node.xmin:node.xmax, :], axis=1), axis=0)/255.0
            colored[ node.ymin:node.ymax, node.xmin:node.xmax ] = clr
        else:
            s,e = node.pixels_interval()
            clr = np.mean(pxflat_image[ node.bpt.pixels[s:e] ], axis=0)/255.0
            flat_colored[ node.bpt.pixels[s:e], :: ] = clr #np.array(cmap[s % len(cmap)])[0:3]
    return colored

def make_segments(nodes, img):
    is_aa = isinstance(nodes[0], shap_bpt.AxisAlignedSegment)
    flat_img = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint32)
    for i, node in enumerate(nodes):
        if is_aa:
            flat_img[ node.ymin:node.ymax, node.xmin:node.xmax ] = i
        else:
            s,e = node.pixels_interval()
            flat_img.ravel()[ node.bpt.pixels[s:e] ] = i
    return flat_img#.reshape((img.shape[0], img.shape[1]))


def mask_from_segments(nodes, img):
    mask = np.zeros(img.shape[:2], dtype=bool)
    for node in nodes:
        node.fill_mask(mask, ascend_hier=False)
    return mask


def boundary_overlay(segment_labels, changed_mask=None, color=(0, 0, 0, 1), mode='thick'):
    transparent = np.zeros((*segment_labels.shape, 4), dtype=np.float32)
    marked = mark_boundaries(transparent, segment_labels, mode=mode, color=color)
    boundary_pixels = np.all(np.isclose(marked, color), axis=-1)
    if changed_mask is not None:
        boundary_pixels &= changed_mask.astype(bool)
    overlay = np.zeros_like(transparent)
    overlay[boundary_pixels] = color
    return overlay


def mask_overlay(mask, color=(1.0, 0.85, 0.05), alpha=0.28):
    overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
    overlay[mask.astype(bool), :3] = color
    overlay[mask.astype(bool), 3] = alpha
    return overlay


def boundary_mask_from_labels(segment_labels, mode='thick'):
    transparent = np.zeros((*segment_labels.shape, 4), dtype=np.float32)
    color = (1, 0, 0, 1)
    marked = mark_boundaries(transparent, segment_labels, mode=mode, color=color)
    return np.all(np.isclose(marked, color), axis=-1)


def dilate_binary_mask(mask, radius=1):
    mask = mask.astype(bool)
    if radius <= 0:
        return mask
    padded = np.pad(mask, radius, mode='constant', constant_values=False)
    out = np.zeros_like(mask, dtype=bool)
    for dy in range(-radius, radius + 1):
        for dx in range(-radius, radius + 1):
            y0 = radius + dy
            x0 = radius + dx
            out |= padded[y0:y0 + mask.shape[0], x0:x0 + mask.shape[1]]
    return out


def new_boundary_overlay(previous_labels, current_labels, color=(1.0, 0.0, 0.0, 1.0), radius=1):
    current_boundary = boundary_mask_from_labels(current_labels, mode='thick')
    if previous_labels is None:
        new_boundary = current_boundary
    else:
        previous_boundary = dilate_binary_mask(boundary_mask_from_labels(previous_labels, mode='thick'), radius=radius)
        new_boundary = current_boundary & ~previous_boundary
    new_boundary = dilate_binary_mask(new_boundary, radius=radius)
    overlay = np.zeros((*current_labels.shape, 4), dtype=np.float32)
    overlay[new_boundary] = color
    return overlay, new_boundary


In [ ]:
np.random.randint(0, 100)

In [ ]:
path_results = os.path.join(original_working_dir, 'results',partition_dir)
# path_results_img = os.path.join(path_results, str(image_no))
os.makedirs(path_results, exist_ok=True)
print(f"Results will be saved in: {path_results}")

In [ ]:
# from examples.scripts.utils_sam import black_rainbow_colormap

def plot_masks(image,masks,mask_types,version='mask',bg_label=0, save_path=None, fontsize=14, destroy_fig=False):
    # print(save_path)
    path_results = '/'.join(save_path.split('/')[:-1])
    image_id = save_path.split('/')[-1].split('_')[0]

    fig, axes = plt.subplots(1,len(masks)+1,figsize=(12 + 2*len(masks), 3))
    axes[0].imshow(image, aspect="auto")
    axes[0].set_title(f'Image: {image_id}', fontsize=fontsize)

    for i, (masks_, mask_type) in enumerate(zip(masks, mask_types)):
        N = np.max(masks_)
        cmap, norm = uts.black_rainbow_colormap(N, rainbow_name="turbo")
        
        bg_ratio = uts.background_ratio(masks_, bg_label=bg_label)
        axx = axes[i+1].imshow(masks_, cmap=cmap, norm=norm, aspect="auto")
        axes[i+1].set_title(f'{mask_type} : {len(np.unique(masks_))} - BG: {bg_ratio["bg_percent"]:.2f}%', fontsize=fontsize)
        colorbar = fig.colorbar(axx, ax=axes[i+1], ticks=np.arange(N + 1))
        colorbar.set_label("Index")
    
    for ax in axes.ravel():
        ax.set_xticks([]) ; ax.set_yticks([])
    plt.tight_layout()

    # print(path_results, image_id)
    plt.savefig(f'{path_results}/{image_id}_image_mask_{version}.png', dpi=150, bbox_inches='tight', pad_inches=0.02)
    if destroy_fig:
        plt.close(fig)
    plt.show()
def plot_coalitions(image,bptrees,save_path=None, fontsize=14,K=8, destroy_fig=False):
    
    leaves = np.zeros(K, dtype=int)
    fig, axes = plt.subplots(len(bptrees), K, figsize=(2 * K, 1.8 * len(bptrees)), squeeze=False)
    # fig.suptitle(f'BPT Partition Expansion: Full Partition with New Boundary Highlighted in Red (K=1..{K})', fontsize=fontsize + 2)
    ## Each row is one BPT construction. Full boundaries stay visible; the newly added boundary is bold red.


    # for ii, image in enumerate(images):
    for ii, (bpt_key,bptree) in enumerate(bptrees.items()):

        bptree = [tree for tree in bptrees.values()][ii]
        base_segment = shap_bpt.BaseSegment()
        # root_node = shap_bpt.AxisAlignedSegment(0, bptree.width, 0, bptree.height, base_segment)
        root_node = shap_bpt.BPT_Segment(bptree, bptree.N-1, base_segment)
        segments = [root_node]
        all_nodes = [root_node]
        
        # axes[0,ii].imshow(image)
        previous_sgm = None
        for jj in range(0,K):
            previous_sgm = make_segments(segments, image)

            # Split all current frontier nodes once. Keep the full partition visible,
            # then emphasize only the newly introduced boundary in red.
            new_segments = []
            for s in segments:
                split = s.split(s, s)
                if split is None:
                    new_segments.append(s)
                    leaves[ii] += 1
                else:
                    new_segments.extend(split)
                    all_nodes.extend(split)

            segments = new_segments

            ax = axes[ii, jj]
            img = colorize(segments, image, 0)
            img = np.clip(0.2 + img * 1.1, 0, 1)
            sgm = make_segments(segments, image)
            all_boundaries = boundary_overlay(
                sgm,
                changed_mask=None,
                color=(0, 0, 0, 0.75),
                mode='thick',
            )
            new_boundary, new_boundary_mask = new_boundary_overlay(
                previous_sgm,
                sgm,
                color=(1.0, 0.0, 0.0, 1.0),
                radius=2,
            )
            ax.imshow(img)
            ax.imshow(all_boundaries)
            ax.imshow(new_boundary)

            if ii == 0:
                ax.set_title(f'Depth\nK={jj + 1}', fontsize=fontsize)
            ax.text(
                0.02,
                0.98,
                f'new={int(new_boundary_mask.sum())}',
                transform=ax.transAxes,
                va='top',
                ha='left',
                fontsize=max(8, fontsize - 4),
                color='white',
                bbox=dict(facecolor='red', alpha=0.90, edgecolor='none', pad=1.5),
            )

        axes[ii, 0].set_ylabel(bpt_key, fontsize=fontsize)
    for ax in axes.ravel():
        ax.set_xticks([]) ; ax.set_yticks([])
    # plt.tight_layout(rect=(0, 0, 1, 0.92))
    plt.subplots_adjust(left=0.05, right=0.95, top=0.85, bottom=0.05, wspace=0.05, hspace=0.05)
    if save_path:
        plt.savefig(f'{save_path}', dpi=150, bbox_inches='tight', pad_inches=0.02)
    if destroy_fig:
        plt.close(fig)
    plt.show()

In [ ]:
# plot_coalitions(image_to_explain,bptrees,masks_sorted, masks_refined, save_path=save_path, fontsize=14, K=8, N=10)

In [ ]:
import sys

scripts_dir = project_root / "examples/scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import utils_xai as utx
import utils_sam as uts

import importlib
importlib.reload(utx) 
importlib.reload(uts)

In [ ]:
def prepare_partitions(partitions, image_shape, max_labels=63):
    capped = uts.cap_partition_labels_local(partitions, max_labels=max_labels)
    return uts.sanitize_partitions_local(capped, image_shape, max_labels=max_labels)

In [ ]:
import json
json_file_path = os.path.join(f'../{partition_dir}', f"{image_id}_results.json")
print('json_file_path:', json_file_path, os.path.exists(json_file_path))
## load json file if it exists, otherwise create a new dictionary
if os.path.exists(json_file_path):
    with open(json_file_path, "r") as f:
        partitions_dict = json.load(f)
else:
    partitions_dict = {}
print('Total Time (sec):', partitions_dict['total_time_sec'])
# print('Number of SAM Masks:', partitions_dict['n_sam_masks'])
print('Number of Coverage Masks:', partitions_dict['n_coverage_masks'])
print('Number of Refined Instances:', partitions_dict['n_refined_instances'])
print('Number of Unique Fillers:', partitions_dict['n_unique_filler'])

In [ ]:
partitions_dict

In [ ]:
# path_partition = 'partitions_24'
# path_partition = 'partitions_100'
partition_dir

In [ ]:
from scipy import ndimage as ndi
import numpy as np

def cap_partition_labels(partitions, max_labels=64):
    partitions = np.asarray(partitions).astype(np.int64)

    labels, counts = np.unique(partitions, return_counts=True)

    if len(labels) <= max_labels:
        keep_labels = labels
    else:
        # Keep the largest regions, merge tiny extras into nearest kept region
        keep_labels = labels[np.argsort(counts)[-max_labels:]]

    keep_mask = np.isin(partitions, keep_labels)

    if not np.all(keep_mask):
        # For every removed pixel, copy nearest kept label
        _, nearest_idx = ndi.distance_transform_edt(~keep_mask, return_indices=True)
        partitions = partitions.copy()
        partitions[~keep_mask] = partitions[tuple(idx[~keep_mask] for idx in nearest_idx)]

    # Remap labels to contiguous 0..K-1
    unique_ids = np.unique(partitions)
    remap = {old: new for new, old in enumerate(unique_ids)}
    partitions = np.vectorize(remap.get)(partitions).astype(np.uint8)

    return partitions


def sanitize_partitions(partitions, image_shape, max_labels=63):
    p = np.asarray(partitions).astype(np.int64, copy=True)

    if p.shape != image_shape[:2]:
        raise ValueError(f"Expected partitions shape {image_shape[:2]}, got {p.shape}")

    p[p < 0] = 0

    labels = [x for x in np.unique(p) if x > 0]
    labels = sorted(labels, key=lambda x: np.sum(p == x), reverse=True)[:max_labels]

    out = np.zeros_like(p, dtype=np.int64)
    for new_label, old_label in enumerate(labels, start=1):
        out[p == old_label] = new_label

    return out

In [ ]:
def load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type='sam'):
    partitions_s = np.load(f"../{partition_dir}/{image_id}_{partition_type}.npy").astype(np.uint16)
    # prepare_partitions(partitions, image_shape, max_labels=63)
    partition_capped_s = cap_partition_labels(partitions_s, max_labels=63)
    partition_capped_s = sanitize_partitions(partition_capped_s, image_to_explain.shape, max_labels=63)

    return partitions_s,partition_capped_s

def load_json_partitions(partition_dir, image_id, verbose=False):
    json_file_path = os.path.join(f'../{partition_dir}', f"{image_id}_results.json")
    if verbose:
        print('json_file_path:', json_file_path, os.path.exists(json_file_path))
    ## load json file if it exists, otherwise create a new dictionary
    if os.path.exists(json_file_path):
        with open(json_file_path, "r") as f:
            return json.load(f)
    else:
        return {}

In [ ]:
partitions, partition_capped = {}, {}
for version in ['sam','coverage','compact', 'filled', 'refined']:
    # partitions[version] = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version, cap_partions=True)
    partitions[version], partition_capped[version] = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version)

In [ ]:
# importlib.reload(uts)
# print('image_id:', image_id )
# uts.plot_masks([partitions['sam'], partitions['coverage']], mask_types=['sam', 'coverage'])
# uts.plot_masks([partition_capped['sam'], partition_capped['coverage']], mask_types=['sam_capped', 'coverage_capped'])


In [ ]:
# importlib.reload(uts)
# uts.plot_masks([partitions['compact'], partitions['filled']], mask_types=['compact', 'Filled'])


In [ ]:
path_results

In [ ]:
importlib.reload(uts)
uts.plot_masks([partitions['sam'], partitions['coverage'], partitions['compact'], partitions['filled'], partitions['refined']], 
               mask_types=['sam', 'coverage', 'compact', 'filled', 'refined'],version='mask', verbose=False)
uts.plot_masks([partition_capped['sam'], partition_capped['coverage'], partition_capped['compact'], partition_capped['filled'], partition_capped['refined']], 
               mask_types=['sam_cap', 'coverage_cap', 'compact_cap', 'filled_cap', 'refined_cap'],version='mask_capped', verbose=False)

## Get ratio of each color and Specially BG Pixels in masks


In [ ]:
importlib.reload(uts)
import pandas as pd
mask_ratios,bgs = {},{}

for version in ['sam','coverage','compact', 'filled', 'refined']:
    mask_ratios[version], bgs[version] = uts.summarize_mask_ratios(partitions[version], f'{version.upper()} Partition', bg_label=0, top_n=12)


display(pd.DataFrame([
        {'mask': 'SAM Sorted', **bgs['sam']},
        {'mask': 'SAM Refined', **bgs['refined']},
        {'mask': 'SAM Coverage', **bgs['coverage']},
        {'mask': 'SAM Compact', **bgs['compact']},
        {'mask': 'SAM Filled', **bgs['filled']},
    ]))
    
# mask_ratios['sam'], bgs['sam'] = uts.summarize_mask_ratios(partitions['sam'], 'SAM Sorted', bg_label=0, top_n=12)

# Optional quick check for the currently loaded image masks.
# if 'masks_sorted' in globals() and 'masks_refined' in globals():
#     mask_ratios['sorted'], bgs['sorted'] = uts.summarize_mask_ratios(masks_sorted, 'SAM Sorted', bg_label=0, top_n=12)
#     mask_ratios['refined'], bgs['refined'] = uts.summarize_mask_ratios(masks_refined, 'SAM Refined', bg_label=0, top_n=12)
#     display(pd.DataFrame([
#         {'mask': 'SAM Sorted', **bgs['sorted']},
#         {'mask': 'SAM Refined', **bgs['refined']},
#     ]))

In [ ]:
partitions,partition_capped = {},{}
for version in ['sam','coverage','compact', 'filled', 'refined']:
    # partitions[version] = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version, cap_partions=True)
    partitions[version],partition_capped[version]  = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version)
    

## Analyze Stats from SAM

In [ ]:
# data_to_csv = []
# label_ratio_rows = []
# verbose = False
# allowed_img_to_plot = 2

# # allowed_imgs = len(image_ids)
# allowed_imgs = 30

# for i, image_id in tqdm(enumerate(image_ids[:allowed_imgs]), total=len(image_ids[:allowed_imgs]), desc="Processing images"):
#     print('=' * 100)
#     image_dir = config["data"]["image_dir"]
#     image_path = os.path.join(image_dir, f'{image_id}.jpg')

#     image_to_explain = load_rgb_image_from_file(image_path)
#     print(image_id)
#     save_path = f"{path_results}/{image_id}_partition_expansion.png"

#     partitions = {}
#     for version in ['sam','coverage','compact', 'filled', 'refined']:
#         partitions[version], partition_capped[version] = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version)
    

In [ ]:
def build_bpt_from_image(img_, partitions, verbose=True):
    bptrees = {}
    bptrees['BPT'] = shap_bpt.build_bpt_from_image(img_)

    for versions in partitions.keys():
        bptrees[versions] = shap_bpt.build_bpt_from_image(img_, prebuilt_partitions=partitions[versions])
        if verbose:
            print(f"Built BPT for {versions} partition: {bptrees[versions]}")
    return bptrees

In [ ]:
bptrees = build_bpt_from_image(image_to_explain, partition_capped)

bptrees

In [ ]:
%%time
save_path = f"{path_results}/{image_id}_partition_expansion.png"
bptrees = build_bpt_from_image(image_to_explain, partition_capped)
plot_masks(image_to_explain,partitions.values(),mask_types=list(partitions.keys()), save_path=save_path,version='mask', fontsize=14)
plot_masks(image_to_explain,partition_capped.values(),mask_types=list(partition_capped.keys()), save_path=save_path,version='mask_capped', fontsize=14)
plot_coalitions(image_to_explain,bptrees,save_path=save_path, fontsize=14,K=10)

## RUN for ALL IMAGES

In [ ]:
import pandas as pd

In [ ]:
# save_path

In [ ]:
data_to_csv = []
label_ratio_rows = []
verbose = False
allowed_img_to_plot = 2

plot__ = False

# allowed_imgs = len(image_ids)
allowed_imgs = 30

for i, image_id in tqdm(enumerate(image_ids[:allowed_imgs]), total=len(image_ids[:allowed_imgs]), desc="Processing images"):
    print('=' * 100)
    image_dir = config["data"]["image_dir"]
    image_path = os.path.join(image_dir, f'{image_id}.jpg')

    image_to_explain = load_rgb_image_from_file(image_path)
    print(image_id)
    save_path = f"{path_results}/{image_id}_partition_expansion.png"

    partitions = {}
    for version in ['sam','coverage','compact', 'filled', 'refined']:
        partitions[version], partition_capped[version] = load_refine_partitions(image_to_explain, partition_dir, image_id, partition_type=version)
    if plot__:
        plot_masks(image_to_explain,partitions.values(),mask_types=list(partitions.keys()),
                version='mask',
                    save_path=save_path, fontsize=14,destroy_fig=True if i == len(image_ids[:allowed_img_to_plot]) - 1 else False)
        plot_masks(image_to_explain,partition_capped.values(),mask_types=list(partition_capped.keys()),
                    version='mask_capped',
                    save_path=save_path, fontsize=14, destroy_fig=True if i == allowed_img_to_plot - 1 else False)

    bptrees = build_bpt_from_image(image_to_explain, partition_capped, verbose=verbose)
    if plot__:
        plot_coalitions(image_to_explain,bptrees,save_path=save_path, fontsize=14,K=10, destroy_fig=True if i == allowed_img_to_plot - 1 else False)
    lms = {}
    bg__ = {}
    for version in ['sam','coverage','compact', 'filled', 'refined']:
        lms[version] = len(np.unique(partitions[version]))
        bg__[version] = uts.background_ratio(partitions[version], bg_label=0)


    for _, row in uts.mask_label_ratios(partitions['sam'], mask_name='SAM Sorted', bg_label=0).iterrows():
        label_ratio_rows.append({'image_id': image_id, **row.to_dict()})
    for _, row in uts.mask_label_ratios(partitions['refined'], mask_name='SAM Refined', bg_label=0).iterrows():
        label_ratio_rows.append({'image_id': image_id, **row.to_dict()})

    for version in ['sam','coverage','compact', 'filled', 'refined']:
        data = {
            'image_id': image_id,
            'mask_type': version,
            'n_masks': lms[version],
            'bg_pixels': bg__[version]['bg_pixels'],
            'bg_ratio': bg__[version]['bg_ratio'],
            'fg_ratio': bg__[version]['fg_ratio'],
            'total_pixels': bg__[version]['total_pixels'],

        }
        data_to_csv.append(data)

        partition_summary_df = pd.DataFrame(data_to_csv)
        partition_summary_df.to_csv(f"{path_results}/partition_summary.csv", index=False)

        label_ratio_df = pd.DataFrame(label_ratio_rows)
        label_ratio_df.to_csv(f"{path_results}/partition_label_ratios.csv", index=False)

display(partition_summary_df)


In [ ]:
partitions_dicts = []
for i, image_id in tqdm(enumerate(image_ids[:allowed_imgs]), total=len(image_ids[:allowed_imgs]), desc="Processing images"):
    if verbose:
        print(f"Processing image {i+1}/{len(image_ids[:allowed_imgs])}: {image_id}")
    # print('=' * 100)
    image_dir = config["data"]["image_dir"]
    image_path = os.path.join(image_dir, f'{image_id}.jpg')

    image_to_explain = load_rgb_image_from_file(image_path)
    if verbose:
        print(image_id)
    save_path = f"{path_results}/{image_id}_partition_expansion.png"
    partitions_dict =  load_json_partitions(partition_dir, image_id)
    partitions_dicts.append(partitions_dict)

partitions_dicts_df = pd.DataFrame(partitions_dicts)


In [ ]:
partitions_dicts_df

## Collect Stats

In [ ]:
df__loaded = pd.read_csv(f"{path_results}/partition_summary.csv")
df__loaded.head(10)


## Create HTML FILE

In [ ]:
from pathlib import Path
import html
import json
import pandas as pd

# Build one HTML report from the saved partition figures and CSV summaries.
report_dir = Path(path_results) if 'path_results' in globals() else Path(
    '/Users/rashid/Nextcloud/RashidPHD/Codes/XAI/ShapBPT-SAM/shap_bpt_sam/examples/notebooks/results/partitions_new_120'
)
report_dir = report_dir.expanduser().resolve()
summary_csv = report_dir / 'partition_summary.csv'
label_ratio_csv = report_dir / 'partition_label_ratios.csv'
html_path = report_dir / 'partition_report.html'

json_dir_candidates = []
if 'partition_dir' in globals():
    json_dir_candidates.extend([Path('..') / partition_dir, Path.cwd() / 'examples' / partition_dir])
try:
    json_dir_candidates.append(report_dir.parents[2] / report_dir.name)
except IndexError:
    pass
json_results_dir = next((p.expanduser().resolve() for p in json_dir_candidates if p.exists()), None)

if not summary_csv.exists():
    raise FileNotFoundError(f'Missing summary CSV: {summary_csv}')

summary_df = pd.read_csv(summary_csv)
label_ratio_df = pd.read_csv(label_ratio_csv) if label_ratio_csv.exists() else pd.DataFrame()

def image_id_str(value):
    text = str(value)
    if text.endswith('.0'):
        text = text[:-2]
    return text.zfill(12)

def fmt_pct(value):
    if pd.isna(value):
        return ''
    return f'{float(value) * 100:.2f}%' if float(value) <= 1.0 else f'{float(value):.2f}%'

def fmt_num(value):
    if pd.isna(value):
        return ''
    return f'{int(value):,}'

def escape(value):
    return html.escape(str(value))

def image_tag(path, label):
    if path.exists():
        return f'<img src="{escape(path.name)}" alt="{escape(label)}" loading="lazy">'
    return f'<div class="missing">Missing image<br>{escape(path.name)}</div>'

json_metric_fields = [
    ('total_time_sec', 'Time sec'),
    ('n_masks', 'JSON masks'),
    ('n_coverage_masks', 'Coverage masks'),
    ('n_refined_instances', 'Refined instances'),
    ('n_unique_filler', 'Unique filler'),
]

def load_json_metrics(image_id):
    if json_results_dir is None:
        return {}
    json_path = json_results_dir / f'{image_id}_results.json'
    if not json_path.exists():
        return {}
    with json_path.open('r') as f:
        return json.load(f)

def fmt_metric(key, value):
    if value is None or pd.isna(value):
        return ''
    if key == 'total_time_sec':
        return f'{float(value):.2f}'
    return fmt_num(value)

summary_df['image_id_str'] = summary_df['image_id'].apply(image_id_str)
if not label_ratio_df.empty:
    label_ratio_df['image_id_str'] = label_ratio_df['image_id'].apply(image_id_str)

mask_counts = summary_df.pivot_table(
    index='image_id_str', columns='mask_type', values='n_masks', aggfunc='first'
).sort_index()
bg_ratios = summary_df.pivot_table(
    index='image_id_str', columns='mask_type', values='bg_ratio', aggfunc='first'
).sort_index()

mask_order = [m for m in ['sam', 'coverage', 'compact', 'filled', 'refined'] if m in mask_counts.columns]
image_ids_report = list(mask_counts.index)

def sortable_header(label, key):
    return (
        f'<th><button class="sort-button" type="button" '
        f'onclick="sortReport(\'{escape(key)}\')">{escape(label)}<span class="sort-indicator"></span></button></th>'
    )

json_metrics_by_image = {image_id: load_json_metrics(image_id) for image_id in image_ids_report}

def sort_attrs_for(image_id):
    attrs = {'image': int(image_id)}
    metrics = json_metrics_by_image.get(image_id, {})
    for key, _ in json_metric_fields:
        value = metrics.get(key)
        attrs[f'json_{key}'] = '' if value is None else value
    for mask in mask_order:
        attrs[f'count_{mask}'] = mask_counts.loc[image_id, mask]
        attrs[f'bg_{mask}'] = bg_ratios.loc[image_id, mask]
    return ' '.join(f'data-sort-{escape(key)}="{escape(value)}"' for key, value in attrs.items())

image_header = sortable_header('Image', 'image')
json_header = ''.join(sortable_header(label, f'json_{key}') for key, label in json_metric_fields)
count_header = ''.join(sortable_header(f'{mask} masks', f'count_{mask}') for mask in mask_order)
bg_header = ''.join(sortable_header(f'{mask} BG', f'bg_{mask}') for mask in mask_order)
overview_rows = []
for image_id in image_ids_report:
    metrics = json_metrics_by_image.get(image_id, {})
    json_cells = ''.join(f'<td>{fmt_metric(key, metrics.get(key))}</td>' for key, _ in json_metric_fields)
    count_cells = ''.join(f'<td>{fmt_num(mask_counts.loc[image_id, mask])}</td>' for mask in mask_order)
    bg_cells = ''.join(f'<td>{fmt_pct(bg_ratios.loc[image_id, mask])}</td>' for mask in mask_order)
    overview_rows.append(
        f'<tr data-image="{image_id}" {sort_attrs_for(image_id)}><td><a href="#{image_id}">{image_id}</a></td>{json_cells}{count_cells}{bg_cells}</tr>'
    )

avg_count_cells = ''.join(
    f'<div><b>{escape(mask)}</b><span>{mask_counts[mask].mean():.1f} masks avg</span></div>'
    for mask in mask_order
)
avg_bg_cells = ''.join(
    f'<div><b>{escape(mask)}</b><span>{(bg_ratios[mask].mean() * 100):.2f}% BG avg</span></div>'
    for mask in mask_order
)

def label_ratio_table(image_id, mask_name):
    if label_ratio_df.empty:
        return '<p class="muted">No label-ratio CSV found.</p>'
    sub = label_ratio_df[(label_ratio_df['image_id_str'] == image_id) & (label_ratio_df['mask'] == mask_name)].copy()
    if sub.empty:
        return '<p class="muted">No label ratios saved for this mask.</p>'

    sub['is_background'] = sub['is_background'].astype(bool)
    sub = sub.sort_values(['is_background', 'pixel_count'], ascending=[False, False])
    rows = []
    for _, row in sub.iterrows():
        cls = ' class="bg-row"' if row['is_background'] else ''
        label = f"{int(row['label'])}"
        if row['is_background']:
            label += ' (BG)'
        rows.append(
            f'<tr{cls}><td>{escape(label)}</td><td>{fmt_num(row["pixel_count"])}</td><td>{float(row["percent"]):.2f}%</td></tr>'
        )
    return (
        '<table class="label-table"><thead><tr><th>Label</th><th>Pixels</th><th>Ratio</th></tr></thead>'
        f'<tbody>{"".join(rows)}</tbody></table>'
    )

image_sections = []
for image_id in image_ids_report:
    metrics_header = '<tr><th>Metric</th>' + ''.join(f'<th>{escape(mask)}</th>' for mask in mask_order) + '</tr>'
    metrics_rows = [
        '<tr><td>No. masks</td>' + ''.join(f'<td>{fmt_num(mask_counts.loc[image_id, mask])}</td>' for mask in mask_order) + '</tr>',
        '<tr><td>BG pixels</td>' + ''.join(f'<td>{fmt_pct(bg_ratios.loc[image_id, mask])}</td>' for mask in mask_order) + '</tr>',
        '<tr><td>FG pixels</td>' + ''.join(f'<td>{fmt_pct(1 - float(bg_ratios.loc[image_id, mask]))}</td>' for mask in mask_order) + '</tr>',
    ]

    image_mask_path = report_dir / f'{image_id}_image_mask_mask.png'
    image_mask_capped_path = report_dir / f'{image_id}_image_mask_mask_capped.png'
    expansion_path = report_dir / f'{image_id}_partition_expansion.png'

    image_sections.append(f'''
<section class="image-card" id="{image_id}" data-image="{image_id}" {sort_attrs_for(image_id)}>
  <div class="section-title">
    <h2>Image {image_id}</h2>
    <a href="#{image_id}">#{image_id}</a>
  </div>
  <table class="metrics-table">
    <thead>{metrics_header}</thead>
    <tbody>{''.join(metrics_rows)}</tbody>
  </table>
  <div class="figure-grid">
    <figure><figcaption>Input and masks</figcaption>{image_tag(image_mask_path, image_id + ' masks')}</figure>
    <figure><figcaption>Input and masks</figcaption>{image_tag(image_mask_capped_path, image_id + ' masks')}</figure>
    <figure><figcaption>BPT partition expansion</figcaption>{image_tag(expansion_path, image_id + ' partition expansion')}</figure>
  </div>
  <details>
    <summary>Per-label pixel ratios for SAM masks</summary>
    <div class="ratio-grid">
      <div><h3>SAM Sorted</h3>{label_ratio_table(image_id, 'SAM Sorted')}</div>
      <div><h3>SAM Refined</h3>{label_ratio_table(image_id, 'SAM Refined')}</div>
    </div>
  </details>
</section>''')

html_doc = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>BPT Partition Report</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 0; color: #1f2933; background: #f7f8fa; }}
  header {{ padding: 24px 32px; background: #111827; color: white; }}
  h1, h2, h3 {{ margin: 0; }}
  header p {{ margin: 8px 0 0; color: #cbd5e1; }}
  main {{ padding: 24px 32px 48px; }}
  .stats {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 12px; margin: 18px 0; }}
  .stat-card {{ background: white; border: 1px solid #e5e7eb; border-radius: 8px; padding: 14px; }}
  .stat-card b {{ display: block; font-size: 22px; margin-bottom: 4px; }}
  .stat-card span {{ color: #64748b; font-size: 13px; }}
  .controls {{ margin: 18px 0; }}
  input {{ width: min(420px, 100%); padding: 10px 12px; border: 1px solid #cbd5e1; border-radius: 8px; font-size: 14px; }}
  .sort-button {{ display: inline-flex; align-items: center; gap: 6px; width: 100%; border: 0; background: transparent; padding: 0; color: inherit; font: inherit; font-weight: 700; text-align: left; cursor: pointer; }}
  .sort-button:hover {{ color: #1d4ed8; }}
  .sort-indicator {{ color: #64748b; font-size: 11px; }}
  table {{ border-collapse: collapse; width: 100%; background: white; }}
  th, td {{ padding: 8px 10px; border-bottom: 1px solid #e5e7eb; text-align: left; font-size: 13px; }}
  th {{ background: #f1f5f9; position: sticky; top: 0; z-index: 1; }}
  a {{ color: #2563eb; text-decoration: none; }}
  .overview-wrap {{ max-height: 420px; overflow: auto; border: 1px solid #e5e7eb; border-radius: 8px; background: white; }}
  .image-card {{ margin-top: 24px; padding: 18px; background: white; border: 1px solid #e5e7eb; border-radius: 8px; }}
  .section-title {{ display: flex; align-items: baseline; justify-content: space-between; gap: 16px; margin-bottom: 12px; }}
  .metrics-table {{ margin-bottom: 16px; }}
  .figure-grid {{ display: grid; grid-template-columns: 1fr; gap: 16px; }}
  figure {{ margin: 0; border: 1px solid #e5e7eb; border-radius: 8px; overflow: hidden; background: #fff; }}
  figcaption {{ padding: 8px 10px; font-weight: 600; background: #f8fafc; border-bottom: 1px solid #e5e7eb; }}
  img {{ display: block; width: 100%; height: auto; }}
  details {{ margin-top: 16px; }}
  summary {{ cursor: pointer; font-weight: 700; color: #334155; }}
  .ratio-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 16px; margin-top: 12px; }}
  .label-table th, .label-table td {{ font-size: 12px; }}
  .bg-row td {{ color: #b91c1c; font-weight: 800; background: #fff1f2; }}
  .missing {{ padding: 32px; text-align: center; color: #b91c1c; background: #fff1f2; }}
  .muted {{ color: #64748b; }}
</style>
</head>
<body>
<header>
  <h1>BPT Partition Report</h1>
  <p>Folder: {escape(report_dir)}</p>
  <p>JSON folder: {escape(json_results_dir) if json_results_dir is not None else "not found"}</p>
</header>
<main>
  <div class="stats">
    <div class="stat-card"><b>{len(image_ids_report)}</b><span>images</span></div>
    <div class="stat-card"><b>{len(mask_order)}</b><span>mask types in summary CSV</span></div>
    <div class="stat-card">{avg_count_cells}</div>
    <div class="stat-card">{avg_bg_cells}</div>
  </div>

  <div class="controls">
    <input id="imageFilter" placeholder="Filter by image id..." oninput="filterImages()">
  </div>

  <h2>All Images Summary</h2>
  <div class="overview-wrap">
    <table>
      <thead><tr>{image_header}{json_header}{count_header}{bg_header}</tr></thead>
      <tbody id="summaryBody">{''.join(overview_rows)}</tbody>
    </table>
  </div>

  <div id="imageSections">{''.join(image_sections)}</div>
</main>
<script>
let activeSort = {{ key: 'image', direction: 'asc' }};

function sortValue(el, key) {{
  const raw = el.getAttribute(`data-sort-${{key}}`) || '';
  const numeric = Number(raw);
  return Number.isNaN(numeric) ? raw : numeric;
}}

function compareValues(a, b, direction) {{
  if (typeof a === 'number' && typeof b === 'number') {{
    return direction === 'asc' ? a - b : b - a;
  }}
  return direction === 'asc'
    ? String(a).localeCompare(String(b))
    : String(b).localeCompare(String(a));
}}

function updateSortIndicators(key, direction) {{
  document.querySelectorAll('.sort-button').forEach(button => {{
    const indicator = button.querySelector('.sort-indicator');
    const isActive = button.getAttribute('onclick').includes(`'${{key}}'`);
    indicator.textContent = isActive ? (direction === 'asc' ? '^' : 'v') : '';
  }});
}}

function sortReport(key) {{
  const direction = activeSort.key === key && activeSort.direction === 'asc' ? 'desc' : 'asc';
  activeSort = {{ key, direction }};

  const tbody = document.getElementById('summaryBody');
  const rows = Array.from(tbody.querySelectorAll('tr'));
  rows.sort((a, b) => compareValues(sortValue(a, key), sortValue(b, key), direction));
  rows.forEach(row => tbody.appendChild(row));

  const imageSections = document.getElementById('imageSections');
  const sections = Array.from(imageSections.querySelectorAll('.image-card'));
  sections.sort((a, b) => compareValues(sortValue(a, key), sortValue(b, key), direction));
  sections.forEach(section => imageSections.appendChild(section));

  updateSortIndicators(key, direction);
}}

function filterImages() {{
  const q = document.getElementById('imageFilter').value.trim();
  document.querySelectorAll('[data-image]').forEach(el => {{
    el.style.display = el.dataset.image.includes(q) ? '' : 'none';
  }});
}}

updateSortIndicators(activeSort.key, activeSort.direction);
</script>
</body>
</html>
'''

html_path.write_text(html_doc, encoding='utf-8')
print(f'HTML report written to: {html_path}')
html_path

In [ ]:
# partition_summary_df = pd.DataFrame(data_to_csv)
# partition_summary_df.to_csv(f"{path_results}/partition_summary.csv", index=False)

# label_ratio_df = pd.DataFrame(label_ratio_rows)
# label_ratio_df.to_csv(f"{path_results}/partition_label_ratios.csv", index=False)

In [ ]:
# partition_summary_df.head(10)


In [ ]:
# bg_ratio_rows = []
# missing_masks = []

# for image_id in tqdm(image_ids):
#     sorted_path = os.path.join('..', path_partition, f'{image_id}_sorted.npy')
#     refined_path = os.path.join('..', path_partition, f'{image_id}_refined.npy')

#     if not os.path.exists(sorted_path) or not os.path.exists(refined_path):
#         missing_masks.append(image_id)
#         continue

#     masks_sorted = np.load(sorted_path).astype(np.uint16)
#     masks_refined = np.load(refined_path).astype(np.uint16)

#     sorted_bg = background_ratio(masks_sorted, bg_label=0)
#     refined_bg = background_ratio(masks_refined, bg_label=0)

#     bg_ratio_rows.append({
#         'image_id': image_id,
#         'sorted_labels': int(len(np.unique(masks_sorted))),
#         'refined_labels': int(len(np.unique(masks_refined))),
#         'sorted_bg_pixels': sorted_bg['bg_pixels'],
#         'sorted_bg_ratio': sorted_bg['bg_ratio'],
#         'sorted_bg_percent': sorted_bg['bg_percent'],
#         'sorted_fg_percent': sorted_bg['fg_percent'],
#         'refined_bg_pixels': refined_bg['bg_pixels'],
#         'refined_bg_ratio': refined_bg['bg_ratio'],
#         'refined_bg_percent': refined_bg['bg_percent'],
#         'refined_fg_percent': refined_bg['fg_percent'],
#         'total_pixels': sorted_bg['total_pixels'],
#     })

# bg_ratio_df = pd.DataFrame(bg_ratio_rows)
# if not bg_ratio_df.empty:
#     bg_ratio_df = bg_ratio_df.sort_values('refined_bg_ratio', ascending=False).reset_index(drop=True)

# os.makedirs(path_results, exist_ok=True)
# bg_ratio_csv = os.path.join(path_results, f'{path_partition}_bg_ratios_all_images.csv')
# bg_ratio_df.to_csv(bg_ratio_csv, index=False)

# print(f'Computed BG ratios for {len(bg_ratio_df)} images')
# print(f'Missing masks: {len(missing_masks)}')
# print(f'Saved BG ratio table: {bg_ratio_csv}')

# display(bg_ratio_df)


In [ ]:
bg_ratio_df[['image_id', 'sorted_bg_percent', 'refined_bg_percent']].head(20)
